# What is actually in these columns

**Module 1 · Session 03, part 1**

The most energetic track in this file is a recording of rain falling on leaves.

That is not a joke about edm. "Rain Forest and Tropical Beach Sound", by an act called Nature
Sounds Nature Music, scores 1.00 on energy, which is the highest value in the dataset. The
least energetic is crickets near a waterfall. Someone put both on a playlist, so Spotify's
audio model measured them exactly the way it measures The Prodigy.

You could have run last week's analysis without ever meeting either of them. Group by genre,
take a mean, write a paragraph about edm, hand it in. The rainforest sits quietly inside the
edm average, and nothing in the pipeline complains.

Today is about the looking that would have caught it.

## The five questions

Work in this order on any table. This notebook goes through them once, on the file you
already know.

| | Question | Where to look |
|---|---|---|
| 1 | How big is it? | `.shape`, `.info()` |
| 2 | What is missing? | `.isna().sum()` |
| 3 | Is anything impossible? | `.describe()`, reading `min` and `max` first |
| 4 | What groups are there, and how big? | `.value_counts()` |
| 5 | What moves with what? | `.corr()`, plus a picture |

Every one of those is a single line of code, which is why they get skipped.


In [ ]:
# Setup. Same file as last week, plus one style block so every chart here matches.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9c9c4", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": "#ececea", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "font.size": 10,
})
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8b8a85"

# The file lives in the class repository. pandas reads a URL exactly like a file.
URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"

songs = pd.read_csv(URL).rename(columns={
    "track_name": "title", "track_artist": "artist",
    "track_popularity": "popularity", "playlist_genre": "genre"})
tracks = songs.drop_duplicates("track_id")   # one row per song instead of one per placement


def listen(rows, *extra_columns):
    """Show rows with a clickable Spotify link. track_id is a real Spotify ID."""
    out = rows[["title", "artist", *extra_columns]].copy()
    out["listen"] = "https://open.spotify.com/track/" + rows["track_id"]
    return HTML(out.to_html(render_links=True, escape=False, index=False))


print(f"Rows (a track on a playlist): {len(songs):,}")
print(f"Distinct songs:               {len(tracks):,}")


Two frames out of one file. `songs` has a row every time a track appears on a playlist, so a
song on five playlists is in there five times. `tracks` has one row per song.

Which one you want depends on what you are claiming. A typical song? Use `tracks`. What is on
these playlists? Use `songs`. Get this wrong and you have not made a coding error, you have
answered a different question.

## 1 and 2. How big, and what is missing


In [ ]:
print("Rows and columns:", songs.shape)

display(songs[["title", "artist", "energy", "danceability", "tempo"]].isna().sum().to_frame("missing"))


Five missing titles and five missing artists, no missing audio. Session 02 settled what to do
about that, which here is nothing: the analysis is about energy, and a row with an unknown
artist still has its energy.

## 3. Is anything impossible?


In [ ]:
display(tracks[["energy", "danceability", "valence", "tempo", "duration_ms"]].describe().round(2))


Read the `min` row first. Four thousand milliseconds is four seconds. A tempo of 0 beats per
minute is not a slow song.

Both of those belong to one record, and it is worth playing in class.


In [ ]:
display(listen(tracks.nsmallest(1, "duration_ms"), "duration_ms", "tempo", "valence"))


Four seconds, no tempo, and a valence of 0.0, which makes it simultaneously the shortest, the
least danceable and the saddest thing in the dataset. It is not a sad song. It is not a song.

Nobody suspected that row. It surfaced from the minimum of two columns that have nothing to do
with each other.


In [ ]:
print("Songs under one minute:", int((tracks["duration_ms"] < 60_000).sum()))
print("Songs with tempo exactly 0:", int((tracks["tempo"] == 0).sum()))
print("Out of:", len(tracks))


Twenty-five short ones out of 28,356. A handful, not a crowd, and far too few to move a mean.

Leaving them in and saying you looked is defensible for this file. Quoting a typical song
length in a contract, you would drop them and say so. The rule changes with the claim, which
is why nobody can write it down for you in advance.

### The extremes are where the measurements give themselves away


In [ ]:
extremes = pd.concat([
    tracks.nlargest(1, "energy"), tracks.nsmallest(1, "energy"),
    tracks.nlargest(1, "instrumentalness"), tracks.nlargest(1, "valence"),
    tracks.nlargest(1, "loudness"),
])
display(listen(extremes, "genre", "energy", "valence"))


Top of the energy scale: rainforest. Bottom: crickets. Most instrumental: waves and wind.
Three of the five extreme values in an audio dataset are not music.

Play one. Then look at the word "energy" again, because it is a signal-processing measure of
loudness, density and noisiness, and it was never a claim about whether a track is exciting.
Students write "edm is the most energetic genre" and mean something the column does not say.

The happiest song in the dataset, at valence 1.00, is "Low Rider" by War, and that one the
model gets right.

## 4. What groups are there, and how big?


In [ ]:
counts = songs["genre"].value_counts()

ax = counts.sort_values().plot(kind="barh", figsize=(7, 3.4), color=BLUE, width=0.7)
ax.set(title="Rows per genre", xlabel="", ylabel="")
ax.bar_label(ax.containers[0], fmt="%,d".replace("%,d", "{:,.0f}").format, padding=4, color="#52514e")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, counts.max() * 1.15)
ax.get_xaxis().set_visible(False)
plt.tight_layout()
plt.show()


Between about 4,900 and 6,000 a side. Unusually even, and not what you normally get.

When it is uneven, a difference between a group of 40 and a group of 40,000 tells you more
about sample sizes than about the world. Print the counts next to every summary. It is a
habit rather than a technique.

## 5. What moves with what?


In [ ]:
correlations = tracks[["energy", "danceability", "loudness", "valence",
                       "acousticness", "popularity"]].corr()
display(correlations["energy"].drop("energy").sort_values(ascending=False).round(3).to_frame("with energy"))


Energy and loudness sit at 0.68. High, and close to a tautology, since both partly measure how
much is going on in the recording.

Energy and danceability sit at -0.08, which is nothing at all. Say that one out loud, because
"energetic" and "danceable" sound like they belong together in English and the data flatly
disagrees.

Picking which of these six numbers deserves a sentence is not something the table does for you.


In [ ]:
sample = tracks.sample(2_000, random_state=2026)   # 28,000 points would be a block of ink

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(sample["loudness"], sample["energy"], s=9, alpha=0.3, color=BLUE, linewidths=0)
ax.set(title="Energy against loudness", xlabel="loudness (dB)", ylabel="energy",
       ylim=(0, 1.02))
ax.text(0.02, 0.96, "2,000 songs, sampled with a fixed seed", transform=ax.transAxes,
        color="#52514e", va="top", fontsize=9)
plt.tight_layout()
plt.show()


The cloud is wide. Loudness will not predict energy for any individual song, which the number
0.68 does not tell you on its own.

And there is a tail of very quiet tracks trailing off to the left. Some of those are the
nature recordings from earlier, sitting in a corner of the plot where no music is.

## Your turn

Pick a numeric column nobody has looked at yet. Plot it, say in one sentence whether its mean
is worth reporting, then find the most extreme song in it and play it.


In [ ]:
# One column, one plot, one sentence, one track.


## One answer, using speechiness


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(tracks["speechiness"], bins=50, color=BLUE, edgecolor="white", linewidth=0.5)
ax.axvline(tracks["speechiness"].mean(), color=ORANGE, linewidth=2)
ax.set(title="speechiness", xlabel="speechiness", ylabel="songs")
ax.text(tracks["speechiness"].mean() + 0.02, ax.get_ylim()[1] * 0.9,
        f"mean {tracks['speechiness'].mean():.2f}", color=ORANGE, fontsize=9)
plt.tight_layout()
plt.show()

print("median:", round(tracks["speechiness"].median(), 3))
display(listen(tracks.nlargest(2, "speechiness"), "speechiness", "genre"))


Most songs sit near zero and a thin tail runs right, so the mean lands at 0.11 while the median
is 0.06. Reporting the mean describes almost nobody.

The extreme tracks are mostly talking, which is what the measure is for. Real records, and they
stay. Skew is not the same thing as error, and students who have just learned about broken rows
tend to want to delete anything unusual.

## Next

Genres differ in energy. Whether any of that difference is worth acting on is a separate
question, and it is the one part 2 spends its time on.
